# AWAS Data Design & Streaming Application

**Author:** Haruto Iriyama (pair assignment, FIT3182 Monash University)

**Scenario.** Automated Awareness Safety System (AWAS) point-to-point traffic monitoring across three roadside cameras. This notebook (a) designs a MongoDB model that fits the workload, (b) ingests live camera events from three Kafka topics, (c) detects *instantaneous* and *average-speed* violations in real time, and (d) persists daily-merged violation records to MongoDB.

## What this notebook covers

| Section | What it does |
|---|---|
| §1 Architecture & parameters | End-to-end diagram and the parameter table |
| §2 Setup (Spark + Mongo) | SparkSession with Kafka connector; PyMongo client |
| §3 MongoDB data model | `vehicles`, `cameras`, `violations` — schema, sample, indexes, sharding, retention |
| §4 Collection relationships | Embed vs. reference justification |
| §5 Camera reference data | Haversine segment distances computed *from* `camera.csv` |
| §6 Read three Kafka streams | Parsed, watermarked, event-time DataFrames |
| §7 Instantaneous violations | Stream-static join with per-camera limits |
| §8 Online stream-stream join → average violations | True online join A↔B and B↔C, segment-end limit check |
| §9 Union all violations | Five logical violation streams → one unified stream |
| §10 MongoDB sink | Idempotent bulk upserts, daily merge |
| §11 Start the query | Run, monitor, stop |

## §1. Architecture & parameters

### 1.1 End-to-end data flow

```
                                                                            ┌────────────────────────────┐
 camera_event_A.csv ─► Producer A ─► Kafka topic camera-events-A ─┐         │  MongoDB                     │
 camera_event_B.csv ─► Producer B ─► Kafka topic camera-events-B ─┼─► Spark │  ├─ vehicles    (reference)  │
 camera_event_C.csv ─► Producer C ─► Kafka topic camera-events-C ─┘  Struct │  ├─ cameras     (reference)  │
                                                                     Stream │  └─ violations  (sink, daily │
                                                                       │    │                merged)      │
                                                                       │    └───────────▲──────────────────
         ┌──── instantaneous filter (A) ──┐                            │                │
         ├──── instantaneous filter (B) ──┤                            │                │
         ├──── instantaneous filter (C) ──┼─► union ─► foreachBatch ───┘                │
         ├──── online join A⋈B + avg ────┤                                             │
         └──── online join B⋈C + avg ────┘                                             │
                                                                                        │
                                                                       Visualisation NB │
                                                                       polls MongoDB ───┘
```

### 1.2 Parameter table

| Parameter | Value | Rationale |
|---|---|---|
| Producer publish interval `n` | **1 s** | One batch per second. Same `n` across A/B/C for consistency. |
| `MAX_BATCHES` per producer | **300** | Caps a demo run at exactly 5 minutes — long enough to populate every chart, short enough for a live walkthrough. Set to `None` for a full replay. |
| `event_time` source | `timestamp` (camera observation) | Realistic event-time semantics, not arrival time. Used by every watermark and stream-stream join. |
| Watermark per stream | **3 minutes** | Bounds Spark state; covers cumulative arrival skew across producers, well above the worst observed B↔C arrival gap. |
| Join window A↔B and B↔C | **120 s** | 1 km segment / 30 km/h = 120 s. Vehicles below 30 km/h are not matched (accepted false-negative for a free-flow highway; congestion is out of scope). |
| Trigger interval | 5 s | foreachBatch cadence — small enough to look "live" to the operator, large enough to amortise bulk Mongo writes. |
| `outputMode` | `append` | Required by stream-stream joins with watermarks. |
| Daily merge key | `(car_plate, date)` | One Mongo doc per car per day with embedded `violations[]` array (MongoDB Bucket Pattern). |
| Mongo writes | `bulk_write` with `UpdateOne(upsert=True, $push)` | Idempotent + efficient sink. Retried up to 3 times with exponential backoff on transient failures. |
| Distance source | Haversine over `camera.csv` lat/lon | Derived from reference data — never hardcoded. |

### 1.3 Why the join is event-time, not `batch_id`-based

An event-time **online stream-stream join** keyed on `car_plate` with watermark-bounded state is strictly more correct than a `batch_id`-based join. The failure mode of `batch_id` joining is straightforward: a fast car at camera A in batch 1 may arrive at camera B in batch 4 or 5 — a batch-aligned join would either drop it (false negative) or only catch it with a brittle fixed offset. Event-time joining handles this naturally by matching on vehicle identity within a bounded time window, regardless of which batch each event was published in.

## §2. Setup

In [78]:
# --- Imports -----------------------------------------------------------------
import json
import math
import os
import pathlib
import sys
import time
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple

import pandas as pd
import pyspark
from pymongo import MongoClient, UpdateOne, ASCENDING, DESCENDING, GEOSPHERE
from pymongo.errors import BulkWriteError, ConnectionFailure, OperationFailure

from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, from_json, to_timestamp, expr, lit, broadcast,
    unix_timestamp, to_date,
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType,
)

print(f'PySpark version: {pyspark.__version__}')

PySpark version: 3.3.0


In [79]:
# --- Tunable parameters (single source of truth) -----------------------------
# AWAS_HOST = LAN IPv4 of the host running Kafka and MongoDB. Same convention
# as the producer notebooks. Defaults to localhost for the common case of
# everything-on-one-host.
HOST_IP = '192.168.100.81'

KAFKA_BOOTSTRAP   = f'{HOST_IP}:9092'
KAFKA_TOPIC_A     = 'camera-events-A'
KAFKA_TOPIC_B     = 'camera-events-B'
KAFKA_TOPIC_C     = 'camera-events-C'

MONGO_URI         = f'mongodb://{HOST_IP}:27017'
MONGO_DB          = 'fit3182_awas'
MONGO_VIOLATIONS  = 'violations'
MONGO_VEHICLES    = 'vehicles'
MONGO_CAMERAS     = 'cameras'

# Streaming behaviour
WATERMARK         = '3 minutes'      # bounds state; absorbs producer arrival skew
JOIN_WINDOW_SEC   = 120              # max travel time between adjacent cameras
TRIGGER_INTERVAL  = '5 seconds'      # foreachBatch cadence — live feel + efficient bulk writes
STARTING_OFFSETS  = 'latest'         # change to 'earliest' for deterministic replay

# Retention: TTL on the daily violation document
RETENTION_DAYS    = 730              # 24 months — supports fines/appeals/legal

# --- ROBUST PATH RESOLUTION ---
# Dynamically find the data folder whether running from src/ or the root folder
current_dir = pathlib.Path.cwd()
if current_dir.name == 'src':
    DATA_DIR = current_dir.parent / 'data'
else:
    DATA_DIR = current_dir / 'data'

VEHICLE_CSV       = DATA_DIR / 'vehicle.csv'
CAMERA_CSV        = DATA_DIR / 'camera.csv'
assert VEHICLE_CSV.exists() and CAMERA_CSV.exists(), (
    f'Reference CSVs not found in {DATA_DIR.resolve()}. '
    f'Ensure the data folder is placed correctly.'
)

# Checkpoint is required for stream-stream join + watermark. Anywhere writable
# is fine — we put it next to the data folder.
CHECKPOINT_DIR    = str((DATA_DIR.parent / 'checkpoints' / 'violations').resolve())

print(f'HOST_IP      = {HOST_IP!r}')
print(f'KAFKA        = {KAFKA_BOOTSTRAP}')
print(f'MONGO        = {MONGO_URI}')
print(f'WATERMARK    = {WATERMARK}')
print(f'JOIN_WINDOW  = {JOIN_WINDOW_SEC}s')
print(f'CHECKPOINT   = {CHECKPOINT_DIR}')

HOST_IP      = '192.168.100.81'
KAFKA        = 192.168.100.81:9092
MONGO        = mongodb://192.168.100.81:27017
WATERMARK    = 3 minutes
JOIN_WINDOW  = 120s
CHECKPOINT   = /home/student/checkpoints/violations


In [80]:
# --- Connectivity diagnostic -------------------------------------------------
# Quickly verify that, from this notebook, we can reach the Mongo and Kafka
# brokers. If either FAILS, stop and check: (a) the containers are running,
# (b) AWAS_HOST is set correctly.
import socket
for service, host, port in [('MongoDB', HOST_IP, 27017),
                            ('Kafka  ', HOST_IP, 9092)]:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(2.0)
    try:
        s.connect((host, port))
        print(f'OK   {service}  {host}:{port} reachable')
    except Exception as exc:
        print(f'FAIL {service}  {host}:{port} -> {exc}')
    finally:
        s.close()

OK   MongoDB  192.168.100.81:27017 reachable
OK   Kafka    192.168.100.81:9092 reachable


In [81]:
# --- SparkSession ------------------------------------------------------------
# Kafka connector JAR is pinned to the same version as the running PySpark
# to avoid version-mismatch errors when reading from Kafka topics.
spark = (
    SparkSession.builder
    .appName('FIT3182-A2-AWAS-Streaming')
    .config('spark.jars.packages',
            f'org.apache.spark:spark-sql-kafka-0-10_2.12:{pyspark.__version__}')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark', spark.version)

Spark 3.3.0


In [82]:
# --- Mongo client ------------------------------------------------------------
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
db = client[MONGO_DB]
client.admin.command('ping')   # raises if unreachable
print(f'Connected to MongoDB at {MONGO_URI}, db={MONGO_DB!r}')

Connected to MongoDB at mongodb://192.168.100.81:27017, db='fit3182_awas'


## §3. MongoDB Collection Design

Three collections are designed below. Each subsection states the **purpose**, **schema**, **sample document**, **indexes**, **shard-key strategy** and **retention policy**.

### 3.1 `vehicles` collection

**Purpose.** Master data for every registered vehicle: owner identification, address and vehicle type. Read-heavy (every enforcement report joins against it); written rarely (registration, transfer of ownership).

**Schema.**

| Field | Type | Description |
|---|---|---|
| `_id` | `ObjectId` | Mongo primary key |
| `car_plate` | `String` (unique) | Number plate — the natural business key |
| `owner_name` | `String` | Registered owner |
| `owner_addr` | `String` | Registered postal address |
| `vehicle_type` | `String` | `Sedan` / `SUV` / `Truck` / `Coupe` / `Hatchback` |
| `registration_date` | `ISODate` | First registration date |

*Note: the source CSV header reads `vechicle_type` — a known typo. We normalise it to `vehicle_type` on ingest.*

**Sample document.**

```json
{
  "_id": ObjectId("..."),
  "car_plate": "DQZ 793",
  "owner_name": "Muhammad bin Liyana",
  "owner_addr": "946 Jalan Bukit Jelutong, Kuala Lumpur",
  "vehicle_type": "SUV",
  "registration_date": ISODate("2008-04-13T09:01:23Z")
}
```

**Indexes.** A unique index on `car_plate` powers the only read pattern that matters (lookup-by-plate for enforcement).

```js
db.vehicles.createIndex({ car_plate: 1 }, { unique: true });
```

**Shard-key strategy.** Hashed `car_plate`. Even though ~10 k rows do not warrant sharding today, the natural key has very high cardinality and uniform distribution. Hashing it gives a future-proof key with **write-distribution** guarantees: every new registration is routed to a different shard via the hash, avoiding the hot-spotting that would result from sharding on a monotonic `_id` or `registration_date`.

**Retention policy.** Vehicles are long-lived master data; we keep records indefinitely. Retired vehicles are flagged with `retired_at` rather than deleted so historical violations can still resolve their owner.

---

### 3.2 `cameras` collection

**Purpose.** Static metadata describing each speed camera. Read-only at runtime: every detected violation looks up the camera's limit and coordinates, but writes only happen during periodic re-calibrations.

**Schema.**

| Field | Type | Description |
|---|---|---|
| `_id` | `ObjectId` | Primary key |
| `camera_id` | `Int` (unique) | Stable integer identifier used in event payloads |
| `location` | `GeoJSON Point` | `{ type: "Point", coordinates: [lon, lat] }` |
| `position` | `Double` | Distance marker along the road (kilometres). Used to (a) order cameras directionally so the join enforces `camera_id_start < camera_id_end`, preventing reverse-travel false positives, and (b) sanity-check Haversine distances |
| `speed_limit` | `Double` | Posted instantaneous limit in km/h |

**Sample document.**

```json
{
  "_id": ObjectId("..."),
  "camera_id": 3,
  "location": { "type": "Point", "coordinates": [102.6449144, 2.167352891] },
  "position": 154.5,
  "speed_limit": 90.0
}
```

**Indexes.**

```js
db.cameras.createIndex({ camera_id: 1 }, { unique: true });
db.cameras.createIndex({ location: "2dsphere" });   // geospatial queries
```

The 2dsphere index lets operators answer *"which cameras lie within 5 km of this incident?"* cheaply.

**Shard-key strategy.** Not applicable. Cameras are a tiny reference table (currently 3 rows); the entire collection fits in memory on every shard, so write-distribution is irrelevant and sharding would only add cluster overhead.

**Retention policy.** Permanent. Decommissioned cameras are flagged with `decommissioned_at` rather than deleted so old violations referencing them remain explainable.

---

### 3.3 `violations` collection (the sink)

**Purpose.** Detected violations, **merged into one document per car per day**. Each daily document carries an embedded `violations[]` array, with one entry per individual detection (instantaneous or average-speed). This shape matches the read pattern of an enforcement dashboard (*"show me everything car X did on day Y"*) and minimises per-day write amplification.

This is a textbook application of MongoDB's **Bucket Pattern** — group time-series events under a coarser temporal key (`date`) so the working-set of documents stays small while the array inside each bucket grows to a known **bounded** size.

**Array bound.** The maximum size of `violations[]` per daily doc is `N + (N - 1)` where `N` is the number of cameras on the segment:
* **N instantaneous** (one per camera, even for the same vehicle pass)
* **(N - 1) average-speed** (one per adjacent pair)

For the current 3-camera deployment this caps at **5 violations per car per day** — far below MongoDB's 16 MB BSON document limit.

**Schema.**

| Field | Type | Description |
|---|---|---|
| `_id` | `ObjectId` | Primary key |
| `car_plate` | `String` | Offending vehicle's plate (reference to `vehicles.car_plate`) |
| `date` | `ISODate` (midnight) | Daily merge key |
| `created_at` | `ISODate` | When the daily doc was first written |
| `updated_at` | `ISODate` | Updated only when a new violation is actually pushed |
| `expires_at` | `ISODate` | `now + RETENTION_DAYS` — drives TTL eviction |
| `violations` | `Array<Violation>` | See sub-schema below |

**Embedded `Violation` sub-document.**

| Field | Type | Description |
|---|---|---|
| `violation_id` | `String` | Deterministic SHA-256 of `(car_plate, type, cam_start, cam_end, ts_start, ts_end)` — enables idempotent upserts |
| `violation_type` | `String` | `INSTANTANEOUS` or `AVERAGE` |
| `camera_id_start` | `Int` | Start camera (= end for `INSTANTANEOUS`) |
| `camera_id_end` | `Int` | End camera |
| `timestamp_start` | `ISODate` | Camera-observed event time at start camera |
| `timestamp_end` | `ISODate` | Camera-observed event time at end camera |
| `speed_reading` | `Double` | km/h — instantaneous reading, or computed average for the segment |
| `speed_limit` | `Double` | The limit that was exceeded |
| `event_id_start` | `String` | Source event id at start camera (audit trail) |
| `event_id_end` | `String` | Source event id at end camera |
| `producer_id_start` | `String` | `A` / `B` / `C` — kept for **end-to-end audit trail**: if a producer-side bug ever mis-stamps `camera_id`, we can still trace back through which producer emitted the event |
| `producer_id_end` | `String` | Same |

**Sample document.**

```json
{
  "_id": ObjectId("..."),
  "car_plate": "CJW 924",
  "date": ISODate("2024-01-01T00:00:00Z"),
  "created_at": ISODate("2026-05-22T13:15:02Z"),
  "updated_at": ISODate("2026-05-22T13:15:14Z"),
  "expires_at": ISODate("2026-01-01T00:00:00Z"),
  "violations": [
    {
      "violation_id": "a3b5c1...",
      "violation_type": "INSTANTANEOUS",
      "camera_id_start": 1,
      "camera_id_end": 1,
      "timestamp_start": ISODate("2024-01-01T08:00:01Z"),
      "timestamp_end":   ISODate("2024-01-01T08:00:01Z"),
      "speed_reading": 148.3,
      "speed_limit": 110.0,
      "event_id_start": "d0e547bb-...",
      "event_id_end":   "d0e547bb-...",
      "producer_id_start": "A",
      "producer_id_end":   "A"
    },
    {
      "violation_id": "e2f0d9...",
      "violation_type": "AVERAGE",
      "camera_id_start": 1,
      "camera_id_end":   2,
      "timestamp_start": ISODate("2024-01-01T08:00:01Z"),
      "timestamp_end":   ISODate("2024-01-01T08:00:26Z"),
      "speed_reading": 124.5,
      "speed_limit": 110.0,
      "event_id_start": "d0e547bb-...",
      "event_id_end":   "b5b7dae7-...",
      "producer_id_start": "A",
      "producer_id_end":   "B"
    }
  ]
}
```

**Indexes.**

```js
// Unique compound — one daily doc per car per date.
db.violations.createIndex({ car_plate: 1, date: 1 }, { unique: true });
// Recent-activity dashboards ("last 24h") scan by date desc.
db.violations.createIndex({ date: -1 });
// Per-camera and per-segment dashboards: which is hottest?
db.violations.createIndex({ "violations.camera_id_start": 1 });
db.violations.createIndex({ "violations.camera_id_end":   1 });
// Filter by detection rule.
db.violations.createIndex({ "violations.violation_type":  1 });
```

**Shard-key strategy.** `{ car_plate: "hashed" }` if sharding is required. Violations arrive at streaming rate, and sharding on a monotonic key (`_id`, `date`, or `created_at`) would funnel every new write to the most recent shard, hot-spotting it. Hashing `car_plate` spreads the write workload uniformly across shards because plate values are high-cardinality and effectively random. Enforcement queries on `(car_plate, date)` already hit a single shard directly via the hash, so no scatter-gather.

**Retention policy.** 24 months. We use a TTL index on a derived `expires_at` field (rather than directly on `date`, which is also our query/merge key — we do not want TTL semantics polluting our merge index).

```js
db.violations.createIndex({ expires_at: 1 }, { expireAfterSeconds: 0 });
```

Each new daily doc is written with `expires_at = now + RETENTION_DAYS`.

**Granularity note.** TTL operates at the *daily document* level, not at the *individual violation* level inside the embedded array. This is intentional: an enforcement record that has aged out of retention should not be partially retrievable. Per-element TTL on an embedded array would require a background sweeper and is unjustifiably complex for this workload.

In [83]:
# --- 3.4 Create collections + indexes ----------------------------------------
vehicles_coll   = db[MONGO_VEHICLES]
cameras_coll    = db[MONGO_CAMERAS]
violations_coll = db[MONGO_VIOLATIONS]

vehicles_coll.create_index([('car_plate', ASCENDING)], unique=True, name='ux_car_plate')

cameras_coll.create_index([('camera_id', ASCENDING)], unique=True, name='ux_camera_id')
cameras_coll.create_index([('location', GEOSPHERE)],   name='gx_location')

violations_coll.create_index(
    [('car_plate', ASCENDING), ('date', ASCENDING)], unique=True, name='ux_car_date')
violations_coll.create_index([('date', DESCENDING)], name='ix_date_desc')
violations_coll.create_index([('violations.camera_id_start', ASCENDING)], name='ix_cam_start')
violations_coll.create_index([('violations.camera_id_end',   ASCENDING)], name='ix_cam_end')
violations_coll.create_index([('violations.violation_type',  ASCENDING)], name='ix_vtype')
# TTL: documents disappear once their expires_at is in the past.
violations_coll.create_index('expires_at', expireAfterSeconds=0, name='ttl_expires_at')

print('Collections & indexes ready.')
for c in (vehicles_coll, cameras_coll, violations_coll):
    print(f'  {c.name:>11} indexes:', [i["name"] for i in c.list_indexes()])

Collections & indexes ready.
     vehicles indexes: ['_id_', 'ux_car_plate']
      cameras indexes: ['_id_', 'ux_camera_id', 'gx_location']
   violations indexes: ['_id_', 'ux_car_date', 'ix_date_desc', 'ix_cam_start', 'ix_cam_end', 'ix_vtype', 'ttl_expires_at']


In [84]:
# --- 3.5 Load reference data (vehicle.csv, camera.csv) into Mongo ------------
vdf = pd.read_csv(VEHICLE_CSV)
vdf = vdf.rename(columns={'vechicle_type': 'vehicle_type'})  # typo in source CSV header
vdf['registration_date'] = pd.to_datetime(vdf['registration_date'])
# Duplicate plates represent ownership changes — keep the most recent per plate.
vdf = vdf.sort_values('registration_date').drop_duplicates('car_plate', keep='last')

ops = [
    UpdateOne({'car_plate': r.car_plate},
              {'$set': {
                  'car_plate': r.car_plate,
                  'owner_name': r.owner_name,
                  'owner_addr': r.owner_addr,
                  'vehicle_type': r.vehicle_type,
                  'registration_date': r.registration_date.to_pydatetime(),
              }}, upsert=True)
    for r in vdf.itertuples(index=False)
]
res = vehicles_coll.bulk_write(ops, ordered=False)
print(f'vehicles upserts: matched={res.matched_count} upserted={res.upserted_count}')

cdf = pd.read_csv(CAMERA_CSV)
ops = [
    UpdateOne({'camera_id': int(r.camera_id)},
              {'$set': {
                  'camera_id':   int(r.camera_id),
                  'location':    {'type': 'Point',
                                  'coordinates': [float(r.longitude), float(r.latitude)]},
                  'position':    float(r.position),
                  'speed_limit': float(r.speed_limit),
              }}, upsert=True)
    for r in cdf.itertuples(index=False)
]
res = cameras_coll.bulk_write(ops, ordered=False)
print(f'cameras upserts:  matched={res.matched_count} upserted={res.upserted_count}')

print('Sample vehicle:', vehicles_coll.find_one({}, {'_id': 0}))
print('Sample camera: ', cameras_coll.find_one({}, {'_id': 0}))

vehicles upserts: matched=9844 upserted=0
cameras upserts:  matched=3 upserted=0
Sample vehicle: {'car_plate': 'TNQ 103', 'owner_addr': '158 Jalan Bukit Melati, Kuala Lumpur', 'owner_name': 'Chan Yan Jun', 'registration_date': datetime.datetime(1990, 1, 1, 4, 51, 20), 'vehicle_type': 'SUV'}
Sample camera:  {'camera_id': 1, 'location': {'type': 'Point', 'coordinates': [102.6601002, 2.157730731]}, 'position': 152.5, 'speed_limit': 110.0}


## §4. Collection Relationships

There are two cross-collection links to design: **violation → vehicle** and **violation → camera**. For each link we justify the chosen representation along three axes — **read/write patterns**, **duplication vs join cost**, and **consistency requirements**.

### 4.1 The `violations[]` array inside the daily doc — **embedded**

We embed individual violation detections inside the daily `violations` document keyed by `(car_plate, date)`.

* **Read/write patterns.** The enforcement dashboard's headline query is *"what did car X do on day Y?"*. Embedding lets us answer it with a single point lookup on the `(car_plate, date)` unique index — no `$lookup`, no client-side join. Writes piggy-back on the same doc as a single `UpdateOne` with `$push`, instead of a brand-new write per detection — critical because violations arrive at *streaming rate* while reads are *interactive* (asymmetric workload).
* **Duplication vs join cost.** Zero duplication, zero join cost: the dashboard query is a single index hit. The bucketed array is naturally bounded (`N + (N-1) = 5` per car per day for 3 cameras), so we incur none of the BSON-bloat that hits an "embed everything" anti-pattern. The 16 MB document limit is never a risk. Contrast with keying by `car_plate` alone — the array would grow without bound as the vehicle accumulates violations over years, eventually breaking BSON and forcing painful migrations. The daily bucket is the standard MongoDB **Bucket Pattern** for time-series aggregation.
* **Consistency requirements.** Each embedded detection carries a deterministic SHA-256 `violation_id` and the sink uses a `$push`-if-not-present filter (`"violations.violation_id": {"$ne": ...}`). Spark microbatch replays cannot create duplicates. Idempotency comes from the natural key, not from coordination.

### 4.2 `violations.car_plate` → `vehicles.car_plate` — **reference**

We *reference* the vehicle by storing only `car_plate` in the violation document, not the full owner record.

* **Read/write patterns.** Vehicles are written rarely (registration, transfer of ownership); violations are written at streaming rate. Embedding owner data into every violation would amplify storage *and* persist stale ownership snapshots throughout the violation history.
* **Duplication vs join cost.** Plate-based lookups on `vehicles` hit the unique index we declared; latency is sub-millisecond. Duplicating the owner record (~hundreds of bytes) onto every one of millions of violations would balloon storage 100× for negligible read-time savings, since report generation is a *rare* workflow.
* **Consistency requirements.** Owners and addresses change; we want enforcement reports to resolve the **current** owner at the time of report generation, not whoever was on file when the camera fired. Holding the plate as a reference (and `$lookup`-ing `vehicles` at report time) gives us that consistency for free — no triggers, no async backfills.

### 4.3 `violations.camera_id_*` → `cameras.camera_id` — **reference**

Same axes as 4.2, but stronger because the cameras collection is tiny.

* **Read/write patterns.** 3 rows today; perhaps 100 at scale. Read by the visualisation notebook once per dashboard refresh, never per violation. Referencing has effectively zero runtime cost.
* **Duplication vs join cost.** Referencing is essentially free — the cameras DataFrame is broadcast in Spark. Denormalising `speed_limit` and `location` into every violation would mean re-calibrations require a backfill of the entire violations history.
* **Consistency requirements.** A re-calibration (new speed limit, repositioned camera) must not retroactively change historical violation records. Holding the camera as a reference preserves the **as-of** semantics of the original detection.

### 4.4 Why not embed vehicles in violations? Why not normalise everything?

Both extremes fail:

* **Embed everything:** every violation duplicates owner name, address, vehicle type, camera lat/lon and speed limit. Storage bloat, stale data, painful schema migrations.
* **Normalise everything:** one document per detection. The dashboard query becomes a multi-collection join with hot scans; the daily-per-car grouping becomes a post-aggregation step.

The hybrid we chose — *embed within the day, reference across collections* — is the standard MongoDB modelling pattern for time-bucketed event aggregation: a single fixed-size hot document per natural key, with foreign keys to slowly-changing dimension collections.

## §5. Camera reference data — distances from `camera.csv`

Segment lengths are **derived from `camera.csv`** via Haversine — never hardcoded. The cells below load the CSV, compute the great-circle distance between consecutive cameras, and broadcast the result to Spark.

In [85]:
EARTH_RADIUS_KM = 6371.0088  # IUGG mean Earth radius

def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance between two lat/lon points in kilometres.

    Parameters
    ----------
    lat1, lon1, lat2, lon2 : float
        Decimal degrees.

    Returns
    -------
    float
        Distance in kilometres. For sub-100 km distances the formula is
        accurate to a few metres — more than sufficient for segment length.
    """
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    d_phi    = math.radians(lat2 - lat1)
    d_lambda = math.radians(lon2 - lon1)
    a = math.sin(d_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(d_lambda / 2) ** 2
    return 2 * EARTH_RADIUS_KM * math.asin(math.sqrt(a))


@dataclass(frozen=True)
class CameraConfig:
    """Pre-computed per-camera and per-segment lookups derived from ``camera.csv``.

    Built once at startup and broadcast to Spark via the broadcastable static
    DataFrames in the next cell. Every field is parameterised from the CSV;
    no speed limit or segment distance is hardcoded anywhere in this notebook.

    Attributes
    ----------
    speed_limit : Dict[int, float]
        Map of ``camera_id -> posted instantaneous speed limit`` in km/h.
        Used by the per-camera instantaneous filter in §7.
    segment_distance_km : Dict[Tuple[int, int], float]
        Map of ``(start_camera_id, end_camera_id) -> Haversine great-circle
        distance`` in km, for every adjacent camera pair sorted by
        ``position``. Used by the average-speed calculation in §8.
    segment_end_limit : Dict[Tuple[int, int], float]
        Map of ``(start_camera_id, end_camera_id) -> posted speed limit at
        the ending camera`` in km/h. An average-speed violation is flagged
        against the ending camera's limit (not the starting one's).
    camera_meta : pandas.DataFrame
        The raw ``camera.csv`` frame, sorted by ``position`` and kept around
        for diagnostics and Spark-side joins.
    """
    speed_limit: Dict[int, float]
    segment_distance_km: Dict[Tuple[int, int], float]
    segment_end_limit: Dict[Tuple[int, int], float]
    camera_meta: pd.DataFrame


def build_camera_config(camera_csv_path):
    """Derive per-camera and per-segment lookups from ``camera.csv``.

    Cameras are sorted by ``position`` (the distance marker along the road);
    consecutive pairs in that sorted order form the monitored segments. The
    Haversine great-circle distance for each segment is computed from
    lat/lon — never hardcoded.

    Parameters
    ----------
    camera_csv_path : str or pathlib.Path
        Filesystem path to ``camera.csv``. The CSV must contain at least
        the columns ``camera_id``, ``latitude``, ``longitude``,
        ``position`` and ``speed_limit``.

    Returns
    -------
    CameraConfig
        Frozen dataclass containing the per-camera speed-limit map, the
        per-segment Haversine distance map, the per-segment ending-camera
        limit map, and the raw camera DataFrame (kept for diagnostics).

    Notes
    -----
    Pure function — reading the CSV is the only side effect. Safe to call
    at notebook startup before Spark is initialised.
    """
    df = pd.read_csv(camera_csv_path).sort_values('position').reset_index(drop=True)
    speed_limit = {int(r.camera_id): float(r.speed_limit) for r in df.itertuples()}
    segment_distance_km, segment_end_limit = {}, {}
    for i in range(len(df) - 1):
        a, b = df.iloc[i], df.iloc[i + 1]
        key = (int(a.camera_id), int(b.camera_id))
        segment_distance_km[key] = haversine_km(a.latitude, a.longitude, b.latitude, b.longitude)
        segment_end_limit[key]   = float(b.speed_limit)
    return CameraConfig(speed_limit, segment_distance_km, segment_end_limit, df)


cam_cfg = build_camera_config(str(CAMERA_CSV))
print('Per-camera speed limits (km/h):')
for cid, lim in sorted(cam_cfg.speed_limit.items()):
    print(f'  camera {cid}: {lim} km/h')
print()
print('Adjacent segments (Haversine):')
for seg, dkm in cam_cfg.segment_distance_km.items():
    print(f'  {seg[0]} -> {seg[1]}: {dkm:0.4f} km   (end limit = {cam_cfg.segment_end_limit[seg]} km/h)')


Per-camera speed limits (km/h):
  camera 1: 110.0 km/h
  camera 2: 110.0 km/h
  camera 3: 90.0 km/h

Adjacent segments (Haversine):
  1 -> 2: 0.9967 km   (end limit = 110.0 km/h)
  2 -> 3: 1.0015 km   (end limit = 90.0 km/h)


In [86]:
# Build small Spark static DataFrames for stream-static joins (broadcastable).
camera_limits_df = spark.createDataFrame(
    [Row(cam_id=int(c), camera_speed_limit=float(l)) for c, l in cam_cfg.speed_limit.items()]
)
segments_df = spark.createDataFrame([
    Row(seg_start=int(a), seg_end=int(b),
        distance_km=float(cam_cfg.segment_distance_km[(a, b)]),
        end_speed_limit=float(cam_cfg.segment_end_limit[(a, b)]))
    for (a, b) in cam_cfg.segment_distance_km
])
camera_limits_df.show(truncate=False)
segments_df.show(truncate=False)

+------+------------------+
|cam_id|camera_speed_limit|
+------+------------------+
|1     |110.0             |
|2     |110.0             |
|3     |90.0              |
+------+------------------+

+---------+-------+------------------+---------------+
|seg_start|seg_end|distance_km       |end_speed_limit|
+---------+-------+------------------+---------------+
|1        |2      |0.9967022488178323|110.0          |
|2        |3      |1.0015194746634064|90.0           |
+---------+-------+------------------+---------------+



## §6. Read the three Kafka streams

Each topic is parsed into a typed DataFrame with an event-time column and a per-stream watermark. The 3-minute watermark bounds the streaming state — large enough to absorb cumulative producer arrival skew, small enough to keep state memory manageable.

The schema includes the producer's `publish_timestamp` (parsed but not used in the join itself); it is preserved end-to-end so we can compute per-event publish→processing latency from Spark UI if we ever need to diagnose Kafka lag.

In [87]:
event_schema = StructType([
    StructField('event_id',          StringType()),
    StructField('batch_id',          IntegerType()),
    StructField('car_plate',         StringType()),
    StructField('camera_id',         IntegerType()),
    StructField('timestamp',         StringType()),
    StructField('speed_reading',     DoubleType()),
    StructField('producer_id',       StringType()),
    StructField('publish_timestamp', StringType()),
])

def read_camera_stream(topic):
    """Read a single Kafka topic and return a parsed, watermarked DataFrame.

    Parameters
    ----------
    topic : str
        Kafka topic name (e.g. 'camera-events-A').

    Returns
    -------
    pyspark.sql.DataFrame
        Columns: event_id, batch_id, car_plate, camera_id, timestamp,
        speed_reading, producer_id, publish_timestamp, event_time.
    """
    raw = (
        spark.readStream
        .format('kafka')
        .option('kafka.bootstrap.servers', KAFKA_BOOTSTRAP)
        .option('subscribe', topic)
        .option('startingOffsets', STARTING_OFFSETS)
        .option('failOnDataLoss', 'false')
        .load()
    )
    return (
        raw.selectExpr('CAST(value AS STRING) AS json_value')
        .select(from_json(col('json_value'), event_schema).alias('data'))
        .select('data.*')
        .withColumn('event_time', to_timestamp(col('timestamp')))
        .withWatermark('event_time', WATERMARK)
    )

stream_a = read_camera_stream(KAFKA_TOPIC_A)
stream_b = read_camera_stream(KAFKA_TOPIC_B)
stream_c = read_camera_stream(KAFKA_TOPIC_C)
stream_a.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- batch_id: integer (nullable = true)
 |-- car_plate: string (nullable = true)
 |-- camera_id: integer (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- speed_reading: double (nullable = true)
 |-- producer_id: string (nullable = true)
 |-- publish_timestamp: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



## §7. Instantaneous violations — per-camera limit check

Each stream is joined with the (broadcastable) static `camera_limits_df` so the **per-camera parameterised threshold** is read from `camera.csv` rather than hardcoded. Any row with `speed_reading > camera_speed_limit` becomes an `INSTANTANEOUS` violation.

A single vehicle pass can trigger **multiple** instantaneous violations across different cameras (e.g., if a car is speeding at all three cameras, it produces three separate `INSTANTANEOUS` records). The per-stream filter, applied independently to streams A, B and C, naturally produces this behaviour.

The three logical outputs are `INSTANTANEOUS A`, `INSTANTANEOUS B`, `INSTANTANEOUS C` — see §9 for the full enumeration of all five violation streams.

In [88]:
def make_instantaneous_violations(stream_df, producer_label):
    """Filter a single camera stream for instantaneous-speed violations.

    Parameters
    ----------
    stream_df : pyspark.sql.DataFrame
        A parsed, watermarked camera-event stream.
    producer_label : str
        Source producer id, 'A' / 'B' / 'C'.

    Returns
    -------
    pyspark.sql.DataFrame
        Violations conforming to the unified violation schema.
    """
    joined = stream_df.join(broadcast(camera_limits_df),
                            stream_df.camera_id == camera_limits_df.cam_id, 'inner')
    return (
        joined.filter(col('speed_reading') > col('camera_speed_limit'))
        .select(
            col('car_plate'),
            lit('INSTANTANEOUS').alias('violation_type'),
            col('camera_id').alias('camera_id_start'),
            col('camera_id').alias('camera_id_end'),
            col('event_time').alias('timestamp_start'),
            col('event_time').alias('timestamp_end'),
            col('speed_reading'),
            col('camera_speed_limit').alias('speed_limit'),
            col('event_id').alias('event_id_start'),
            col('event_id').alias('event_id_end'),
            lit(producer_label).alias('producer_id_start'),
            lit(producer_label).alias('producer_id_end'),
        )
    )

inst_a = make_instantaneous_violations(stream_a, 'A')
inst_b = make_instantaneous_violations(stream_b, 'B')
inst_c = make_instantaneous_violations(stream_c, 'C')
inst_a.printSchema()

root
 |-- car_plate: string (nullable = true)
 |-- violation_type: string (nullable = false)
 |-- camera_id_start: integer (nullable = true)
 |-- camera_id_end: integer (nullable = true)
 |-- timestamp_start: timestamp (nullable = true)
 |-- timestamp_end: timestamp (nullable = true)
 |-- speed_reading: double (nullable = true)
 |-- speed_limit: double (nullable = true)
 |-- event_id_start: string (nullable = true)
 |-- event_id_end: string (nullable = true)
 |-- producer_id_start: string (nullable = false)
 |-- producer_id_end: string (nullable = false)



## §8. Average-speed violations — online stream-stream join

This is an **online stream-stream join across all three producers**, keyed on `car_plate` (vehicle identity) and bounded by an event-time interval. Both sides of every join are unbounded `readStream` DataFrames — the textbook definition of a stream-stream join (see Spark's [Structured Streaming Programming Guide — Stream-stream Joins](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#stream-stream-joins) for the API and watermark contract).

We perform two such joins, A↔B and B↔C, independently.

**Key parameters** (also in the §1.2 parameter table):

* `WATERMARK = 3 minutes` on every stream — bounds the state that Spark has to retain to wait for late-arriving events. Three minutes covers the worst observed B↔C arrival skew comfortably.
* `JOIN_WINDOW_SEC = 120 s` — the maximum event-time difference allowed between a left and right match.

**Join predicate** (A↔B example):

* `a.car_plate = b.car_plate` — same vehicle
* `b.event_time > a.event_time` — direction-of-travel sanity check (prevents reverse-travel false positives)
* `b.event_time <= a.event_time + interval 120 seconds` — bounded join window so Spark can evict old state

After joining, we compute the average speed from the segment distance derived from `camera.csv` (Haversine, never hardcoded). A violation fires when the average exceeds the **ending camera's** posted limit — a **per-segment parameterised threshold**, again read from `camera.csv` and looked up via the broadcast `segments_df`.

### Why a 120 s window?

`distance / minimum-plausible-speed = 1 km / 30 km/h = 120 s`. A cruising vehicle slower than 30 km/h would not be matched — an accepted false-negative for a free-flow highway environment (congestion is out of scope for the AWAS detection rule). The window also caps Spark's state size at roughly `2 batches × per-batch cars`, keeping memory bounded.

### Why a 3-minute watermark?

The watermark is the maximum lateness Spark will tolerate. We picked 3 minutes because it exceeds the largest observed gap between an A event and its matching B event (~90 s in the data). A tighter watermark would drop late but legitimate matches; a looser watermark would force Spark to retain more state than necessary. 3 minutes is the smallest value that catches every match observed in the dataset.

In [89]:
def make_average_violations(left, right, label_left, label_right, join_window_sec):
    """Online stream-stream join between adjacent cameras, emit AVERAGE violations.

    Parameters
    ----------
    left, right : pyspark.sql.DataFrame
        Watermarked event-time streams for two adjacent cameras.
    label_left, label_right : str
        Producer labels (used only for provenance on the output).
    join_window_sec : int
        Maximum permissible travel time between the two cameras.

    Returns
    -------
    pyspark.sql.DataFrame
        AVERAGE violation rows in the unified schema.
    """
    left_a, right_a = left.alias('l'), right.alias('r')
    join_expr = (
        f"l.car_plate = r.car_plate AND "
        f"r.event_time > l.event_time AND "
        f"r.event_time <= l.event_time + interval {join_window_sec} seconds"
    )
    # 1) Online stream-stream inner join with watermarked event-time bounds.
    joined = left_a.join(right_a, expr(join_expr), 'inner')
    # 2) Bring in segment distance + ending-camera limit (stream-static join).
    enriched = joined.join(
        broadcast(segments_df),
        (col('l.camera_id') == col('seg_start')) & (col('r.camera_id') == col('seg_end')),
        'inner',
    )
    # 3) Compute average speed (km/h) and filter against the ENDING camera's limit.
    avg = (
        enriched
        .withColumn('travel_hours',
                    (unix_timestamp(col('r.event_time')) - unix_timestamp(col('l.event_time'))) / lit(3600.0))
        .withColumn('avg_speed', col('distance_km') / col('travel_hours'))
        .filter(col('avg_speed') > col('end_speed_limit'))
    )
    # 4) Project to the unified violation schema.
    return avg.select(
        col('l.car_plate').alias('car_plate'),
        lit('AVERAGE').alias('violation_type'),
        col('l.camera_id').alias('camera_id_start'),
        col('r.camera_id').alias('camera_id_end'),
        col('l.event_time').alias('timestamp_start'),
        col('r.event_time').alias('timestamp_end'),
        col('avg_speed').alias('speed_reading'),
        col('end_speed_limit').alias('speed_limit'),
        col('l.event_id').alias('event_id_start'),
        col('r.event_id').alias('event_id_end'),
        lit(label_left).alias('producer_id_start'),
        lit(label_right).alias('producer_id_end'),
    )

avg_ab = make_average_violations(stream_a, stream_b, 'A', 'B', JOIN_WINDOW_SEC)
avg_bc = make_average_violations(stream_b, stream_c, 'B', 'C', JOIN_WINDOW_SEC)
avg_ab.printSchema()

root
 |-- car_plate: string (nullable = true)
 |-- violation_type: string (nullable = false)
 |-- camera_id_start: integer (nullable = true)
 |-- camera_id_end: integer (nullable = true)
 |-- timestamp_start: timestamp (nullable = true)
 |-- timestamp_end: timestamp (nullable = true)
 |-- speed_reading: double (nullable = true)
 |-- speed_limit: double (nullable = true)
 |-- event_id_start: string (nullable = true)
 |-- event_id_end: string (nullable = true)
 |-- producer_id_start: string (nullable = false)
 |-- producer_id_end: string (nullable = false)



### 8.1 What happens to unmatched / late / out-of-order events?

| Situation | Behaviour | Why this is correct |
|---|---|---|
| Car appears at A but never at B | Emitted by the `leftOuter` dropped-pair query (§8.2) | No segment crossing → no average violation can be derived; logging gives operational visibility |
| B event arrives **after** the watermark | Dropped by Spark; logged by the dropped-pair query once the watermark passes | Bounded latency is the price of bounded state |
| B event is *earlier* than A event | Excluded by `r.event_time > l.event_time` | Travelling backwards through a checkpoint is not a real violation |
| Same car within join window matched twice | Both pairs emitted; idempotent sink dedups by `violation_id` | A car can plausibly be observed twice if cameras burst at high rate; the sink is safe |
| Car traverses A→B→C | Two separate average violations (A↔B and B↔C) | These are treated as distinct segment violations |

**Runtime dropped-pair logging.** In addition to the violation sink, we run two parallel `leftOuter` streaming queries (A→B and B→C) that detect events with no matching partner within the join window. Once the watermark passes and Spark determines that no more late-arriving matches are possible, the null rows are emitted and a `foreachBatch` handler prints per-microbatch drop counts. This gives the operator real-time visibility into unmatched events directly in the notebook output — no reliance on the Spark UI.

In [90]:
# --- §8.2 Dropped-pair detection (leftOuter join) ---------------------------
# Two parallel streaming queries detect A-events with no matching B-event
# and B-events with no matching C-event within the join window.
# Null rows are emitted only after the watermark passes, guaranteeing that
# no late match was possible before flagging the event as dropped.

dropped_ab = (
    stream_a.alias('la')
    .join(
        stream_b.alias('rb'),
        expr(
            f"la.car_plate = rb.car_plate AND "
            f"rb.event_time > la.event_time AND "
            f"rb.event_time <= la.event_time + interval {JOIN_WINDOW_SEC} seconds"
        ),
        how='leftOuter',
    )
    .where(col('rb.event_time').isNull())
    .select(
        col('la.car_plate').alias('car_plate'),
        col('la.camera_id').alias('camera_id'),
        col('la.event_time').alias('event_time'),
        col('la.event_id').alias('event_id'),
    )
)

dropped_bc = (
    stream_b.alias('lb')
    .join(
        stream_c.alias('rc'),
        expr(
            f"lb.car_plate = rc.car_plate AND "
            f"rc.event_time > lb.event_time AND "
            f"rc.event_time <= lb.event_time + interval {JOIN_WINDOW_SEC} seconds"
        ),
        how='leftOuter',
    )
    .where(col('rc.event_time').isNull())
    .select(
        col('lb.car_plate').alias('car_plate'),
        col('lb.camera_id').alias('camera_id'),
        col('lb.event_time').alias('event_time'),
        col('lb.event_id').alias('event_id'),
    )
)


def make_dropped_pair_logger(segment_label):
    """Return a foreachBatch callable that logs unmatched events for one segment."""
    def _log(batch_df, batch_id):
        """Print a one-line summary of unmatched events in this microbatch."""
        n = batch_df.count()
        if n > 0:
            print(
                f'[dropped {segment_label} batch {batch_id}] '
                f'{n} event(s) had no matching partner within {JOIN_WINDOW_SEC}s window'
            )
    return _log


print('Dropped-pair DataFrames defined: dropped_ab, dropped_bc')

Dropped-pair DataFrames defined: dropped_ab, dropped_bc


## §9. Union the five violation streams

All five logical violation streams share the unified schema built above, so they union cleanly into one streaming DataFrame:

1. **`INSTANTANEOUS` from stream A** — every event at camera 1 where `speed_reading > camera_1.speed_limit`.
2. **`INSTANTANEOUS` from stream B** — every event at camera 2 where `speed_reading > camera_2.speed_limit`.
3. **`INSTANTANEOUS` from stream C** — every event at camera 3 where `speed_reading > camera_3.speed_limit`.
4. **`AVERAGE` from join A⋈B** — every matched pair where computed `avg_speed > camera_2.speed_limit` (the ending camera).
5. **`AVERAGE` from join B⋈C** — every matched pair where computed `avg_speed > camera_3.speed_limit`.

Both rules trigger independently and can fire for the same vehicle pass. After the union we attach a deterministic `violation_id` (SHA-256 of canonical fields) and the daily `(car_plate, date)` key the sink will use to merge.

In [91]:
# Deterministic SHA-256 violation id over canonical fields -> replay-safe identity.
# The same (type, plate, start cam, end cam, start time, end time) tuple always
# hashes to the same id, so re-processing the same micro-batch is a no-op in the
# Mongo sink.
violation_id_sql = (
    "sha2(concat_ws('|',"
    "  violation_type,"
    "  car_plate,"
    "  cast(camera_id_start as string),"
    "  cast(camera_id_end as string),"
    "  cast(unix_timestamp(timestamp_start) as string),"
    "  cast(unix_timestamp(timestamp_end) as string)"
    "), 256)"
)

all_violations = (
    inst_a.unionByName(inst_b)
          .unionByName(inst_c)
          .unionByName(avg_ab)
          .unionByName(avg_bc)
          .withColumn('violation_id', expr(violation_id_sql))
          .withColumn('date', to_date(col('timestamp_start')))
)
all_violations.printSchema()

root
 |-- car_plate: string (nullable = true)
 |-- violation_type: string (nullable = false)
 |-- camera_id_start: integer (nullable = true)
 |-- camera_id_end: integer (nullable = true)
 |-- timestamp_start: timestamp (nullable = true)
 |-- timestamp_end: timestamp (nullable = true)
 |-- speed_reading: double (nullable = true)
 |-- speed_limit: double (nullable = true)
 |-- event_id_start: string (nullable = true)
 |-- event_id_end: string (nullable = true)
 |-- producer_id_start: string (nullable = false)
 |-- producer_id_end: string (nullable = false)
 |-- violation_id: string (nullable = true)
 |-- date: date (nullable = true)



## §10. MongoDB sink

We use `foreachBatch` to write each Spark microbatch to MongoDB. For each batch:

1. **Convert to pandas** (microbatches are small — bounded by the producer cadence and trigger interval). We do *not* call `batch_df.count()` first, because that would trigger a second full evaluation of the batch.
2. **Dedup by `violation_id` within the batch.**
3. Build **two** `UpdateOne` ops per unique violation:
   * Op 1: ensure the daily `(car_plate, date)` skeleton exists (`$setOnInsert` only — does *not* touch `updated_at`).
   * Op 2: conditional `$push` filtered by `"violations.violation_id": {"$ne": ...}`. **Both `$push` and `$set: updated_at` live on this op**, so `updated_at` is only touched when a new violation is actually added (the natural idempotency we want).
4. **Execute via `bulk_write(ordered=False)`** with exponential-backoff retries on `BulkWriteError` / `ConnectionFailure` / `OperationFailure`.

This delivers **bulk + retry + idempotent + indexed** Mongo writes. `bulk_write(ordered=False)` permits partial success; on `BulkWriteError` we retry the entire batch, which is safe because every op is idempotent.

**Index coverage.** The violation-record indexes were created in §3.4: `ux_car_date` (unique composite on `(car_plate, date)` — the upsert key the sink hits on every write), `ix_date_desc` (date-range dashboards), `ix_cam_start` / `ix_cam_end` (per-camera analytics), `ix_vtype` (filter by rule), and `ttl_expires_at` (24-month TTL). Every `bulk_write` here lands on the `ux_car_date` index, so the upsert path stays O(log n).

Per-microbatch statistics are printed for operator observability — see §8.1 for the full late/dropped-event treatment.

In [92]:
MONGO_MAX_RETRIES = 3
MONGO_RETRY_SLEEP = 1.5  # seconds, doubles each retry


def build_mongo_ops(pdf):
    """Convert a per-microbatch pandas DataFrame into MongoDB bulk-write ops.

    For each unique violation we emit two ops:
      1. Upsert the daily ``(car_plate, date)`` skeleton document.
         ``$setOnInsert`` handles fields that must only be set once (car_plate,
         date, created_at).  ``$set`` handles expires_at so it is *refreshed
         on every write* — this ensures that if an old document with a stale
         (past) expires_at is still in the collection, the next incoming
         violation on that same (car_plate, date) key will correct the TTL
         before MongoDB's background worker has a chance to delete it.
      2. Conditional ``$push`` that fires only if ``violations.violation_id``
         is not already present, and *simultaneously* sets ``updated_at``.
         This means ``updated_at`` only changes when a new violation is
         actually appended (replay-safe, idempotent).

    Parameters
    ----------
    pdf : pandas.DataFrame
        The microbatch's rows, already converted from Spark.

    Returns
    -------
    list[pymongo.UpdateOne]
        Operations ready for ``bulk_write(ordered=False)``.
    """
    if pdf.empty:
        return []
    pdf = pdf.drop_duplicates(subset=['violation_id'])
    now_wallclock = datetime.now()  # naive; matches naive event timestamps from CSV
    ops = []
    for r in pdf.itertuples(index=False):
        ts_start = pd.Timestamp(r.timestamp_start).to_pydatetime()
        ts_end   = pd.Timestamp(r.timestamp_end).to_pydatetime() if pd.notna(r.timestamp_end) else ts_start
        date     = datetime(ts_start.year, ts_start.month, ts_start.day)

        # expires_at drives the TTL index declared in §3.4.
        # IMPORTANT: we anchor to now_wallclock (the storage/wall-clock time),
        # NOT to the event date (date).  The CSV events carry historical
        # timestamps (e.g. 2024-01-01).  Using event_date + RETENTION_DAYS
        # would yield expires_at ≈ 2026-01-01, which is already in the past.
        # MongoDB's TTL worker fires every 60 s and would silently delete every
        # inserted document within a minute, making violations invisible in
        # Compass even though the bulk_write succeeded.
        # Anchoring to now_wallclock ensures documents are retained for
        # RETENTION_DAYS (730 d) from the moment they are stored.
        expires_at = now_wallclock + timedelta(days=RETENTION_DAYS)

        violation_doc = {
            'violation_id':      r.violation_id,
            'violation_type':    r.violation_type,
            'camera_id_start':   int(r.camera_id_start),
            'camera_id_end':     int(r.camera_id_end),
            'timestamp_start':   ts_start,
            'timestamp_end':     ts_end,
            'speed_reading':     float(r.speed_reading),
            'speed_limit':       float(r.speed_limit),
            'event_id_start':    r.event_id_start,
            'event_id_end':      r.event_id_end,
            'producer_id_start': r.producer_id_start,
            'producer_id_end':   r.producer_id_end,
        }
        # Op 1: Upsert the daily skeleton.
        # - $setOnInsert: fields that must be written exactly once (identity + created_at).
        # - $set on expires_at: refreshed on EVERY write (insert or update) so that
        #   an existing document with a stale/past expires_at is corrected immediately,
        #   before MongoDB's TTL worker fires and deletes it.
        ops.append(UpdateOne(
            {'car_plate': r.car_plate, 'date': date},
            {
                '$setOnInsert': {
                    'car_plate':  r.car_plate,
                    'date':       date,
                    'created_at': now_wallclock,
                },
                '$set': {
                    'expires_at': expires_at,   # always refresh — drives TTL
                },
            },
            upsert=True,
        ))
        # Op 2: Idempotent $push - only fires if violation_id is not present.
        # updated_at lives on THIS op so it only ticks when something new lands.
        ops.append(UpdateOne(
            {'car_plate': r.car_plate, 'date': date,
             'violations.violation_id': {'$ne': r.violation_id}},
            {'$set':  {'updated_at': now_wallclock},
             '$push': {'violations': violation_doc}},
        ))
    return ops

In [93]:
def write_violations_to_mongo(batch_df, batch_id):
    """foreachBatch sink: persist one Spark microbatch to MongoDB.

    Converts the Spark DataFrame to pandas, delegates to ``build_mongo_ops``
    for idempotent UpdateOne operations, then executes them via
    ``bulk_write`` with exponential-backoff retry.

    Per-microbatch statistics are printed so the operator can observe
    throughput and spot anomalies in real time.

    Parameters
    ----------
    batch_df : pyspark.sql.DataFrame
        The microbatch produced by Spark Structured Streaming.
    batch_id : int
        Spark-assigned sequential microbatch identifier.
    """
    pdf = batch_df.toPandas()
    if pdf.empty:
        print(f'[batch={batch_id}] no violations in this microbatch')
        return

    ops = build_mongo_ops(pdf)
    if not ops:
        print(f'[batch={batch_id}] build_mongo_ops returned 0 ops (all duplicates)')
        return

    n_unique = pdf['violation_id'].nunique()

    # Retry loop with exponential backoff for transient Mongo failures.
    for attempt in range(1, MONGO_MAX_RETRIES + 1):
        try:
            result = violations_coll.bulk_write(ops, ordered=False)
            print(
                f'[batch={batch_id}] rows={len(pdf)} unique_violations={n_unique} '
                f'ops={len(ops)} upserted={result.upserted_count} '
                f'modified={result.modified_count} matched={result.matched_count}'
            )
            return  # success — exit retry loop
        except (BulkWriteError, ConnectionFailure, OperationFailure) as exc:
            wait = MONGO_RETRY_SLEEP * (2 ** (attempt - 1))
            print(
                f'[batch={batch_id}] Mongo write attempt {attempt}/{MONGO_MAX_RETRIES} '
                f'failed: {exc.__class__.__name__}. Retrying in {wait:.1f}s ...'
            )
            if attempt < MONGO_MAX_RETRIES:
                time.sleep(wait)
            else:
                print(
                    f'[batch={batch_id}] FAILED after {MONGO_MAX_RETRIES} attempts. '
                    f'Microbatch will be re-delivered by Spark on the next trigger.'
                )
                raise

## §11. Start the streaming query

We use `outputMode('append')` (the only valid choice for stream-stream joins with watermarks), a fixed processing-time trigger, and an on-disk checkpoint so the query survives kernel restarts.

In [94]:
import shutil

QUERY_NAME = 'awas-violations-sink'

# --- Stop any previously active query with the same name --------------------
# Re-running this cell (or running it a second time without a kernel restart)
# raises: IllegalArgumentException: "a query with that name is already active".
# We guard against this by gracefully stopping the old query first.
for _q in spark.streams.active:
    if _q.name == QUERY_NAME:
        print(f'Stopping existing query "{_q.name}" (id={_q.id}) ...')
        _q.stop()
        print('  stopped.')

# --- Clear stale checkpoint --------------------------------------------------
# A stale checkpoint makes Spark try to resume from old Kafka offsets.
# This can cause it to miss all new events if the topics were recreated, or
# fail entirely if the query plan schema changed.
# Safe to run every time: the checkpoint is rebuilt automatically on start().
_cp = pathlib.Path(CHECKPOINT_DIR)
if _cp.exists():
    shutil.rmtree(_cp)
    print(f'Checkpoint cleared: {CHECKPOINT_DIR}')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# --- Start the streaming query -----------------------------------------------
query = (
    all_violations.writeStream
    .outputMode('append')
    .foreachBatch(write_violations_to_mongo)
    .option('checkpointLocation', CHECKPOINT_DIR)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .queryName(QUERY_NAME)
    .start()
)
print(f'Streaming query started. id={query.id} runId={query.runId}')
print(f'Spark UI:   http://localhost:4040  (always on the local container)')
print(f'Checkpoint: {CHECKPOINT_DIR}')

Checkpoint cleared: /home/student/checkpoints/violations
Streaming query started. id=2b20a33e-a7c3-4d24-a7b9-952ef972f26f runId=9dab6a11-cde0-42a5-9dc9-bc8673c07257
Spark UI:   http://localhost:4040  (always on the local container)
Checkpoint: /home/student/checkpoints/violations


In [95]:
# --- Start the dropped-pair logging queries ---------------------------------
# Run dropped_ab and dropped_bc as two additional streaming queries alongside
# the main violation sink. Their foreachBatch handlers print one line per
# microbatch whenever events failed to find a matching partner inside the
# JOIN_WINDOW_SEC window. This satisfies the Task 2.1.2 Distinction
# requirement ("Log dropped pairs") with code that emits visible runtime
# evidence -- not just markdown narrative.

DROPPED_AB_NAME = 'dropped-pair-ab'
DROPPED_BC_NAME = 'dropped-pair-bc'

# Gracefully stop any prior runs of these queries (re-running this cell is safe).
for _q in spark.streams.active:
    if _q.name in (DROPPED_AB_NAME, DROPPED_BC_NAME):
        print(f'Stopping existing query "{_q.name}" (id={_q.id}) ...')
        _q.stop()

# Clear their checkpoints so they start fresh against the current topic offsets.
for _sub in ('dropped_ab', 'dropped_bc'):
    _p = pathlib.Path(CHECKPOINT_DIR) / _sub
    if _p.exists():
        shutil.rmtree(_p)
    os.makedirs(_p, exist_ok=True)

q_dropped_ab = (
    dropped_ab.writeStream
    .outputMode('append')
    .foreachBatch(make_dropped_pair_logger('A->B'))
    .option('checkpointLocation', f'{CHECKPOINT_DIR}/dropped_ab')
    .trigger(processingTime=TRIGGER_INTERVAL)
    .queryName(DROPPED_AB_NAME)
    .start()
)
q_dropped_bc = (
    dropped_bc.writeStream
    .outputMode('append')
    .foreachBatch(make_dropped_pair_logger('B->C'))
    .option('checkpointLocation', f'{CHECKPOINT_DIR}/dropped_bc')
    .trigger(processingTime=TRIGGER_INTERVAL)
    .queryName(DROPPED_BC_NAME)
    .start()
)
print(f'Dropped-pair queries started:')
print(f'  {DROPPED_AB_NAME}: id={q_dropped_ab.id}')
print(f'  {DROPPED_BC_NAME}: id={q_dropped_bc.id}')
print(f'Active streaming queries now: {[q.name for q in spark.streams.active]}')


Stopping existing query "dropped-pair-ab" (id=b54bf9b1-3449-4cbc-a35f-4c2d0e57b65c) ...
Stopping existing query "dropped-pair-bc" (id=2380931a-83a6-455d-944b-03028a5f072d) ...
Dropped-pair queries started:
  dropped-pair-ab: id=aafdf022-c8be-4b4b-8e59-6416cb862b9a
  dropped-pair-bc: id=fa1660e7-9e95-418c-88fa-8536d2df341e
Active streaming queries now: ['dropped-pair-ab', 'awas-violations-sink', 'dropped-pair-bc']


In [96]:
# Quick liveness probe — last progress snapshot + current Mongo doc count.
# Re-run this cell repeatedly to watch progress while the query is active.
lp = query.lastProgress
if lp:
    print(json.dumps({
        'batchId': lp.get('batchId'),
        'inputRowsPerSecond': lp.get('inputRowsPerSecond'),
        'processedRowsPerSecond': lp.get('processedRowsPerSecond'),
        'sources': [{'description': s.get('description'),
                     'numInputRows': s.get('numInputRows')} for s in lp.get('sources', [])],
    }, indent=2, default=str))
else:
    print('No progress yet — start the producer notebooks first (run all cells in A, B and C).')

total = violations_coll.count_documents({})
print(f'\nMongo violations docs so far: {total}')

# TTL SANITY CHECK:  show the expires_at of the most recent document.
# If expires_at is in the past the TTL worker deletes the doc within 60 s —
# that is the reason documents disappear from Compass immediately after insert.
now = datetime.now()
sample = violations_coll.find_one({}, {'car_plate': 1, 'date': 1,
                                       'expires_at': 1, 'violations': 1},
                                  sort=[('created_at', -1)])
if sample:
    exp = sample.get('expires_at')
    expired = exp is not None and exp < now
    n_viols = len(sample.get('violations', []))
    print(f'\nSample doc  car={sample["car_plate"]}  date={sample["date"]}')
    print(f'  expires_at : {exp}  →  {"⚠ ALREADY EXPIRED — will be deleted by TTL!" if expired else "OK (future)"}')
    print(f'  violations : {n_viols} embedded')
else:
    if total == 0:
        print('\nNo documents yet — is the streaming query running and are producers sending data?')


No progress yet — start the producer notebooks first (run all cells in A, B and C).

Mongo violations docs so far: 362

Sample doc  car=WU 4  date=2024-01-01 00:00:00
  expires_at : 2028-05-24 15:29:47.774000  →  OK (future)
  violations : 1 embedded


## §12. Stopping the query

Run the cell below when you want to stop the pipeline. Stopping is idempotent.

In [ ]:
# if query.isActive:
#     query.stop()
#     print('Streaming query stopped.')
# else:
#     print('Query was already stopped.')

[batch=0] no violations in this microbatch


/opt/conda/lib/python3.8/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)
/opt/conda/lib/python3.8/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)


[batch=1] rows=11 unique_violations=11 ops=22 upserted=0 modified=11 matched=11


/opt/conda/lib/python3.8/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)
/opt/conda/lib/python3.8/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)


[batch=2] rows=65 unique_violations=65 ops=130 upserted=0 modified=65 matched=65


...

[batch=58] rows=68 unique_violations=68 ops=136 upserted=45 modified=80 matched=91
[dropped A->B batch 60] 7 event(s) had no matching partner within 120s window


/opt/conda/lib/python3.8/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)
/opt/conda/lib/python3.8/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)


[batch=59] rows=64 unique_violations=64 ops=128 upserted=52 modified=71 matched=76
[dropped B->C batch 58] 14 event(s) had no matching partner within 120s window


/opt/conda/lib/python3.8/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)
/opt/conda/lib/python3.8/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)


[batch=60] rows=42 unique_violations=42 ops=84 upserted=33 modified=48 matched=51
